In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from transformers import (
    AutoTokenizer, 
    AutoImageProcessor, 
    ViTModel, 
    Wav2Vec2Model, 
    AutoFeatureExtractor
)
from datasets import load_dataset
from accelerate import Accelerator
import math

#  MULTI-STREAM INGESTOR 
class GeminiDataStreamer(IterableDataset):
    def __init__(self, mode="sft"):
        self.mode = mode
        self.tokenizer = AutoTokenizer.from_pretrained("gpt2")
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.v_proc = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224", use_fast=True)
        self.a_proc = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")
        
        # Primary Multimodal Streams
        self.v_stream = load_dataset("HuggingFaceH4/llava-instruct-mix-vsft", split="train", streaming=True)
        self.a_stream = load_dataset("PolyAI/minds14", "en-US", split="train", streaming=True)
        
        if mode == "dpo":
            # Using Argilla's DPO pairs 
            self.dpo_stream = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train", streaming=True)

    def _extract_text(self, item):
        if isinstance(item, str): return item
        if isinstance(item, list) and len(item) > 0:
            if isinstance(item[0], dict) and 'text' in item[0]:
                return " ".join([i['text'] for i in item if 'text' in i])
        return str(item)

    def __iter__(self):
        v_iter = iter(self.v_stream)
        a_iter = iter(self.a_stream)
        d_iter = iter(self.dpo_stream) if self.mode == "dpo" else None
        
        while True:
            try:
                v_item = next(v_iter)
                a_item = next(a_iter)
                
                # Image Prep
                img = v_item.get('image') or (v_item.get('images')[0] if v_item.get('images') else None)
                if img is None: continue
                pixels = self.v_proc(images=img.convert("RGB"), return_tensors="pt")['pixel_values'].squeeze(0)
                
                # Audio Prep
                audio = self.a_proc(a_item['audio']['array'], sampling_rate=16000, return_tensors="pt")['input_values'].squeeze(0)

                if self.mode == "sft":
                    # Extract SFT Text
                    raw_text = ""
                    if 'messages' in v_item:
                        raw_text = next((m['content'] for m in reversed(v_item['messages']) if m['role'] == 'assistant'), "")
                    elif 'conversations' in v_item:
                        raw_text = v_item['conversations'][-1]['value']
                    
                    text_str = self._extract_text(raw_text)
                    if not text_str: continue
                    
                    tokens = self.tokenizer(text_str, max_length=128, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    yield pixels, tokens, audio
                
                else: # DPO MODE
                    d_item = next(d_iter)
                    # Chosen/Rejected from Orca DPO
                    c_txt = d_item['chosen']
                    r_txt = d_item['rejected']
                    
                    c_tokens = self.tokenizer(c_txt, max_length=128, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    r_tokens = self.tokenizer(r_txt, max_length=128, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    
                    yield pixels, c_tokens, r_tokens, audio

            except StopIteration: break
            except Exception: continue

# --- 2.  ARCHITECTURE ---
class SparseMoE(nn.Module):
    def __init__(self, d_model, num_experts=4):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model*2), nn.SiLU(), nn.Linear(d_model*2, d_model)) 
            for _ in range(num_experts)
        ])

    def forward(self, x):
        b, s, d = x.shape
        flat_x = x.view(-1, d)
        probs = F.softmax(self.gate(flat_x), dim=-1)
        weights, idx = torch.topk(probs, 1, dim=-1)
        
        out = torch.zeros_like(flat_x)
        for i, expert in enumerate(self.experts):
            mask = (idx == i).squeeze()
            if mask.any():
                out[mask] = expert(flat_x[mask]) * weights[mask]
        return out.view(b, s, d), probs.mean(0).std()

class GeminiLite(nn.Module):
    def __init__(self, vocab_size, d_model=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")
        self.audio_enc = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        self.v_proj = nn.Linear(768, d_model)
        self.a_proj = nn.Linear(768, d_model)
        self.moe = SparseMoE(d_model)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, tokens, pixels, audio):
        t_feat = self.token_emb(tokens)
        with torch.no_grad():
            v_feat = self.v_proj(self.vit(pixels).last_hidden_state)
            a_feat = self.a_proj(self.audio_enc(audio).last_hidden_state)
        
        x = torch.cat([a_feat, v_feat, t_feat], dim=1)
        x_moe, aux_loss = self.moe(self.norm(x))
        return self.head(x + x_moe), aux_loss

# 3. EXECUTION 
def train():
    acc = Accelerator()
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    model = GeminiLite(len(tokenizer))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
    
    # SFT
    print(">>> Phase 1: SFT...")
    sft_loader = DataLoader(GeminiDataStreamer(mode="sft"), batch_size=1)
    model, opt, sft_loader = acc.prepare(model, opt, sft_loader)
    
    model.train()
    for i, (p, t, a) in enumerate(sft_loader):
        opt.zero_grad()
        logits, aux = model(t, p, a)
        loss = F.cross_entropy(logits[:, -128:-1, :].reshape(-1, len(tokenizer)), t[:, 1:].reshape(-1))
        acc.backward(loss + 0.01 * aux)
        opt.step()
        if i % 2 == 0: print(f"SFT {i} | Loss: {loss.item():.4f}")
        if i >= 400: break

    # DPO
    print("\n>>> Phase 2: DPO Alignment...")
    dpo_loader = DataLoader(GeminiDataStreamer(mode="dpo"), batch_size=1)
    dpo_loader = acc.prepare(dpo_loader)
    
    for i, (p, c, r, a) in enumerate(dpo_loader):
        opt.zero_grad()
        l_c, _ = model(c, p, a)
        l_r, _ = model(r, p, a)
        
        log_p_c = F.log_softmax(l_c, -1).mean()
        log_p_r = F.log_softmax(l_r, -1).mean()
        # DPO Loss Beta = 0.1
        dpo_loss = -F.logsigmoid(0.1 * (log_p_c - log_p_r))
        
        acc.backward(dpo_loss)
        opt.step()
        if i % 2 == 0: print(f"DPO {i} | Loss: {dpo_loss.item():.4f}")
        if i >= 200: break

    print(">>> Pipeline Finished Successfully.")
    print(">>> Pipeline Finished Successfully.")

    
    trainable_weights = {
        "v_proj": model.v_proj.state_dict(),
        "a_proj": model.a_proj.state_dict(),
        "moe": model.moe.state_dict(),
        "head": model.head.state_dict()
    }
    torch.save(trainable_weights, "gemini_adapters.pth")
    print("Multimodal adapters saved to gemini_adapters.pth")
if __name__ == "__main__":
    train()

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image
import requests
from transformers import AutoTokenizer, AutoImageProcessor, ViTModel, Wav2Vec2Model, AutoFeatureExtractor

class SparseMoE(nn.Module):
    def __init__(self, d_model, num_experts=4):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model*2), nn.SiLU(), nn.Linear(d_model*2, d_model)) 
            for _ in range(num_experts)
        ])

    def forward(self, x):
        b, s, d = x.shape
        flat_x = x.view(-1, d)
        probs = F.softmax(self.gate(flat_x), dim=-1)
        weights, idx = torch.topk(probs, 1, dim=-1)
        
        out = torch.zeros_like(flat_x)
        for i, expert in enumerate(self.experts):
            mask = (idx == i).squeeze()
            if mask.any():
                out[mask] = expert(flat_x[mask]) * weights[mask]
        return out.view(b, s, d), probs.mean(0).std()

class GeminiLite(nn.Module):
    def __init__(self, vocab_size, d_model=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")
        self.audio_enc = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        self.v_proj = nn.Linear(768, d_model)
        self.a_proj = nn.Linear(768, d_model)
        self.moe = SparseMoE(d_model)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, tokens, pixels, audio):
        t_feat = self.token_emb(tokens)
        with torch.no_grad():
            v_feat = self.v_proj(self.vit(pixels).last_hidden_state)
            a_feat = self.a_proj(self.audio_enc(audio).last_hidden_state)
        
        x = torch.cat([a_feat, v_feat, t_feat], dim=1)
        x_moe, aux_loss = self.moe(self.norm(x))
        return self.head(x + x_moe), aux_loss


# 2. INITIALIZE GLOBAL MODEL
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = GeminiLite(len(tokenizer))

# Load the adapter dictionary
checkpoint = torch.load("/kaggle/working/gemini_adapters.pth")

# Map the saved weights to the model's internal modules
model.v_proj.load_state_dict(checkpoint['v_proj'])
model.a_proj.load_state_dict(checkpoint['a_proj'])
model.moe.load_state_dict(checkpoint['moe'])
model.head.load_state_dict(checkpoint['head'])

print("Successfully loaded multimodal adapters!")
model.eval()
# 3.GENERATION FUNCTION
@torch.no_grad()
def generate_response(model, question, image_path_or_url, max_new_tokens=20):
    v_proc = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
    a_proc = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")

    # Load Image
    if image_path_or_url.startswith("http"):
        img = Image.open(requests.get(image_path_or_url, stream=True).raw).convert("RGB")
    else:
        img = Image.open(image_path_or_url).convert("RGB")
    
    pixels = v_proc(images=img, return_tensors="pt")['pixel_values']
    audio = a_proc(torch.zeros(16000), sampling_rate=16000, return_tensors="pt")['input_values']
    
    # Encode Question
    input_ids = tokenizer.encode(question, return_tensors="pt")
    generated = input_ids

    for _ in range(max_new_tokens):
        logits, _ = model(generated, pixels, audio)
        next_token_logits = logits[:, -1, :]
        
        # Apply a bit of temperature for variety
        probs = F.softmax(next_token_logits / 0.8, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        generated = torch.cat([generated, next_token], dim=1)
        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated[0], skip_special_tokens=True)

# 4. RUN INFERENCE
test_image = "/kaggle/input/datasets/anwarshamim/images-for-test/images.jpg"
test_query = "explain this"

print("Generating...")
answer = generate_response(model, test_query, test_image)
print("-" * 30)
print(f"GEMINI LITE SAYS: {answer}")

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from transformers import (
    AutoTokenizer, 
    AutoImageProcessor, 
    ViTModel, 
    Wav2Vec2Model, 
    AutoFeatureExtractor,
    AutoModelForCausalLM 
)
from datasets import load_dataset
from accelerate import Accelerator, dispatch_model, infer_auto_device_map
from PIL import Image
import requests

# Set the global LLM choice
LLM_BACKBONE = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" 

# ==========================================
# 1. MULTI-STREAM INGESTOR
# ==========================================
class GeminiDataStreamer(IterableDataset):
    def __init__(self, mode="sft", max_length=128, max_audio_sec=5):
        self.mode = mode
        self.max_length = max_length
        self.max_audio_length = 16000 * max_audio_sec 
        
        self.tokenizer = AutoTokenizer.from_pretrained(LLM_BACKBONE)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        self.v_proc = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224", use_fast=True)
        self.a_proc = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")
        
        self.v_stream = load_dataset("HuggingFaceH4/llava-instruct-mix-vsft", split="train", streaming=True)
        self.a_stream = load_dataset("PolyAI/minds14", "en-US", split="train", streaming=True)
        
        if mode == "dpo":
            self.dpo_stream = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train", streaming=True)

    def _extract_text(self, item):
        if isinstance(item, str): return item
        if isinstance(item, list) and len(item) > 0:
            if isinstance(item[0], dict) and 'text' in item[0]:
                return " ".join([i['text'] for i in item if 'text' in i])
        return str(item)

    def __iter__(self):
        v_iter = iter(self.v_stream)
        a_iter = iter(self.a_stream)
        d_iter = iter(self.dpo_stream) if self.mode == "dpo" else None
        
        while True:
            try:
                v_item = next(v_iter)
                a_item = next(a_iter)
                
                # Image
                img = v_item.get('image') or (v_item.get('images')[0] if v_item.get('images') else None)
                if img is None: continue
                pixels = self.v_proc(images=img.convert("RGB"), return_tensors="pt")['pixel_values'].squeeze(0)
                
                # Audio (Padded to 5 seconds perfectly)
                audio_features = self.a_proc(
                    a_item['audio']['array'], 
                    sampling_rate=16000, 
                    padding="max_length", 
                    max_length=self.max_audio_length, 
                    truncation=True,
                    return_tensors="pt"
                )
                audio = audio_features['input_values'].squeeze(0)

                if self.mode == "sft":
                    raw_text = ""
                    if 'messages' in v_item:
                        raw_text = next((m['content'] for m in reversed(v_item['messages']) if m['role'] == 'assistant'), "")
                    elif 'conversations' in v_item:
                        raw_text = v_item['conversations'][-1]['value']
                    
                    text_str = self._extract_text(raw_text)
                    if not text_str: continue
                    
                    tokens = self.tokenizer(text_str, max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    yield pixels, tokens, audio
                
                else: # DPO
                    d_item = next(d_iter)
                    c_txt = d_item['chosen']
                    r_txt = d_item['rejected']
                    
                    c_tokens = self.tokenizer(c_txt, max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    r_tokens = self.tokenizer(r_txt, max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    
                    yield pixels, c_tokens, r_tokens, audio

            except StopIteration: break
            except Exception: continue


# ==========================================
# 2. SOTA ARCHITECTURE
# ==========================================
class SparseMoE(nn.Module):
    def __init__(self, d_model, num_experts=4):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model*2), nn.SiLU(), nn.Linear(d_model*2, d_model)) 
            for _ in range(num_experts)
        ])

    def forward(self, x):
        b, s, d = x.shape
        flat_x = x.view(-1, d)
        probs = F.softmax(self.gate(flat_x), dim=-1)
        weights, idx = torch.topk(probs, 1, dim=-1)
        
        out = torch.zeros_like(flat_x)
        for i, expert in enumerate(self.experts):
            mask = (idx == i).squeeze()
            if mask.any():
                out[mask] = expert(flat_x[mask]) * weights[mask]
        return out.view(b, s, d), probs.mean(0).std()


class GeminiLiteSOTA(nn.Module):
    def __init__(self, llm_name=LLM_BACKBONE, compute_dtype=torch.bfloat16):
        super().__init__()
        self.compute_dtype = compute_dtype
        
        # 1. BRAIN: Force Load in BFloat16 natively
        self.llm = AutoModelForCausalLM.from_pretrained(llm_name, torch_dtype=compute_dtype)
        d_model = self.llm.config.hidden_size
        
        # 2. SENSORS: Force Load in BFloat16 natively
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224", torch_dtype=compute_dtype)
        self.audio_enc = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base", torch_dtype=compute_dtype)
        
        self.llm.requires_grad_(False)
        self.vit.requires_grad_(False)
        self.audio_enc.requires_grad_(False)
        
        # 3. ADAPTERS
        self.v_proj = nn.Linear(self.vit.config.hidden_size, d_model, dtype=compute_dtype)
        self.a_proj = nn.Linear(self.audio_enc.config.hidden_size, d_model, dtype=compute_dtype)
        self.moe = SparseMoE(d_model).to(compute_dtype)
        self.norm = nn.LayerNorm(d_model, dtype=compute_dtype)

    def forward(self, tokens, pixels, audio, pad_token_id):
        # Cast inputs to BFloat16 to avoid collision
        pixels = pixels.to(self.compute_dtype)
        audio = audio.to(self.compute_dtype)
        
        with torch.no_grad():
            v_feat = self.vit(pixels).last_hidden_state
            a_feat = self.audio_enc(audio).last_hidden_state
        
        # Get text embeddings 
        t_embeds = self.llm.get_input_embeddings()(tokens).to(self.compute_dtype)
        
        v_embeds = self.v_proj(v_feat)
        a_embeds = self.a_proj(a_feat)
        
        mm_embeds = torch.cat([a_embeds, v_embeds], dim=1)
        mm_moe_out, aux_loss = self.moe(self.norm(mm_embeds))
        mm_embeds = mm_embeds + mm_moe_out 
        
        inputs_embeds = torch.cat([mm_embeds, t_embeds], dim=1)
        
        # SOTA FIX: Dynamic Attention Mask
        # We must tell the LLM to ignore the <pad> tokens in the text portion!
        batch_size = tokens.size(0)
        mm_len = mm_embeds.size(1)
        
        # Image and Audio are always attended to (1s)
        mm_mask = torch.ones((batch_size, mm_len), dtype=torch.long, device=tokens.device)
        # Text mask: 1 if it's a real word, 0 if it's padding
        text_mask = (tokens != pad_token_id).long()
        
        attention_mask = torch.cat([mm_mask, text_mask], dim=1)
        
        # Pass both embeddings and mask to LLM
        outputs = self.llm(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
        
        return outputs.logits, aux_loss, mm_len


# ==========================================
# 3. LOSS FUNCTIONS
# ==========================================
def compute_causal_loss(logits, tokens, mm_len, pad_token_id):
    labels = tokens.clone()
    labels[labels == pad_token_id] = -100
    
    batch_size = tokens.size(0)
    mm_labels = torch.full((batch_size, mm_len), -100, dtype=torch.long, device=tokens.device)
    full_labels = torch.cat([mm_labels, labels], dim=1)
    
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = full_labels[..., 1:].contiguous()
    
    return F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

def get_batch_logps(logits, tokens, mm_len, pad_token_id):
    labels = tokens.clone()
    labels[labels == pad_token_id] = -100
    mm_labels = torch.full((tokens.size(0), mm_len), -100, dtype=torch.long, device=tokens.device)
    full_labels = torch.cat([mm_labels, labels], dim=1)
    
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = full_labels[..., 1:].contiguous()
    
    log_probs = F.log_softmax(shift_logits, dim=-1)
    loss_mask = shift_labels != -100
    shift_labels[shift_labels == -100] = 0 
    
    token_log_probs = torch.gather(log_probs, dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)
    return (token_log_probs * loss_mask).sum(dim=-1)


# ==========================================
# 4. TRAINING EXECUTION
# ==========================================
def train():
    # Force accelerate to use bfloat16 to match our model
    acc = Accelerator(mixed_precision="bf16")
    tokenizer = AutoTokenizer.from_pretrained(LLM_BACKBONE)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = GeminiLiteSOTA(compute_dtype=torch.bfloat16)
    
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable_params, lr=1e-4)
    
    print(">>> Phase 1: SFT (Multimodal Alignment)...")
    sft_loader = DataLoader(GeminiDataStreamer(mode="sft"), batch_size=2)
    model, opt, sft_loader = acc.prepare(model, opt, sft_loader)
    
    model.train()
    for i, (p, t, a) in enumerate(sft_loader):
        opt.zero_grad()
        
        # Pass pad_token_id so the model can build the attention mask
        logits, aux_loss, mm_len = model(t, p, a, tokenizer.pad_token_id)
        
        loss = compute_causal_loss(logits, t, mm_len, tokenizer.pad_token_id)
        
        total_loss = loss + 0.01 * aux_loss
        acc.backward(total_loss)
        opt.step()
        
        if i % 10 == 0: 
            print(f"SFT Step {i} | CrossEntropy Loss: {loss.item():.4f}")
        if i >= 200: break

    print("\n>>> Phase 2: DPO (Preference Alignment)...")
    dpo_loader = DataLoader(GeminiDataStreamer(mode="dpo"), batch_size=1)
    dpo_loader = acc.prepare(dpo_loader)
    
    for i, (p, c_t, r_t, a) in enumerate(dpo_loader):
        opt.zero_grad()
        
        logits_c, _, mm_len = model(c_t, p, a, tokenizer.pad_token_id)
        logits_r, _, _ = model(r_t, p, a, tokenizer.pad_token_id)
        
        l_c = get_batch_logps(logits_c, c_t, mm_len, tokenizer.pad_token_id)
        l_r = get_batch_logps(logits_r, r_t, mm_len, tokenizer.pad_token_id)
        
        dpo_loss = -F.logsigmoid(0.1 * (l_c - l_r)).mean()
        
        acc.backward(dpo_loss)
        opt.step()
        if i % 10 == 0: 
            print(f"DPO Step {i} | Loss: {dpo_loss.item():.4f}")
        if i >= 100: break

    print("\n>>> Training Finished Successfully!")
    
    acc.wait_for_everyone()
    if acc.is_main_process:
        unwrap_model = acc.unwrap_model(model)
        trainable_weights = {
            "v_proj": unwrap_model.v_proj.state_dict(),
            "a_proj": unwrap_model.a_proj.state_dict(),
            "moe": unwrap_model.moe.state_dict(),
        }
        torch.save(trainable_weights, "gemini_adapters.pth")
        print("Multimodal adapters saved to gemini_adapters.pth")


# ==========================================
# 5. MULTI-GPU INFERENCE FUNCTION
# ==========================================
@torch.no_grad()
def generate_response(model, tokenizer, question, image_path_or_url, max_new_tokens=30):
    model.eval()
    
    # Identify device of model (Handles multi-gpu automatically)
    device = next(model.parameters()).device
    compute_dtype = model.compute_dtype
    
    v_proc = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
    a_proc = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")

    if image_path_or_url.startswith("http"):
        img = Image.open(requests.get(image_path_or_url, stream=True).raw).convert("RGB")
    else:
        img = Image.open(image_path_or_url).convert("RGB")
    
    pixels = v_proc(images=img, return_tensors="pt")['pixel_values'].to(device, dtype=compute_dtype)
    audio = a_proc(torch.zeros(16000 * 5), sampling_rate=16000, return_tensors="pt")['input_values'].to(device, dtype=compute_dtype)
    
    prompt = f"<|user|>\n<image>\n{question}</s>\n<|assistant|>\n"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    v_feat = model.vit(pixels).last_hidden_state
    a_feat = model.audio_enc(audio).last_hidden_state
    v_embeds = model.v_proj(v_feat)
    a_embeds = model.a_proj(a_feat)
    
    mm_embeds = torch.cat([a_embeds, v_embeds], dim=1)
    mm_moe_out, _ = model.moe(model.norm(mm_embeds))
    mm_embeds = mm_embeds + mm_moe_out
    
    t_embeds = model.llm.get_input_embeddings()(input_ids).to(compute_dtype)
    inputs_embeds = torch.cat([mm_embeds, t_embeds], dim=1)
    
    # We must also generate the attention mask for inference
    mm_mask = torch.ones((1, mm_embeds.size(1)), dtype=torch.long, device=device)
    t_mask = torch.ones((1, input_ids.size(1)), dtype=torch.long, device=device)
    attention_mask = torch.cat([mm_mask, t_mask], dim=1)
    
    generated_ids = model.llm.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.7,
        do_sample=True
    )
    
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)


if __name__ == "__main__":
    train()
    
    print("\n--- Running Multi-GPU Inference Test ---")
    tok = AutoTokenizer.from_pretrained(LLM_BACKBONE)
    
    # 1. Load the model framework in bfloat16 to CPU first
    test_model = GeminiLiteSOTA(compute_dtype=torch.bfloat16)
    
    # 2. Load the trained adapters
    checkpoint = torch.load("gemini_adapters.pth", map_location="cpu")
    test_model.v_proj.load_state_dict(checkpoint['v_proj'])
    test_model.a_proj.load_state_dict(checkpoint['a_proj'])
    test_model.moe.load_state_dict(checkpoint['moe'])
    
    # 3. SOTA Multi-GPU Dispatch: Automatically splits memory evenly across your 2 GPUs!
    device_map = infer_auto_device_map(test_model, max_memory={0: "14GiB", 1: "14GiB"})
    test_model = dispatch_model(test_model, device_map=device_map)
    
    test_image = "https://images.unsplash.com/photo-1543373014-cfe4f4bc1cdf?q=80&w=300&auto=format&fit=crop"
    
    answer = generate_response(test_model, tok, "Describe this image:", test_image)
    print(f"\nMODEL RESPONSE:\n{answer}")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.



MODEL RESPONSE:
or    :o m ,  d : in  - ,  ,":tet .. 
" or" ," in ,


In [ ]:
    test_image = "/kaggle/input/datasets/anwarshamim/images-for-test/images.jpg"
    
    answer = generate_response(test_model, tok, "Describe this image:", test_image)
    print(f"\nMODEL RESPONSE:\n{answer}")

In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from transformers import (
    AutoTokenizer, 
    AutoImageProcessor, 
    ViTModel, 
    Wav2Vec2Model, 
    AutoFeatureExtractor,
    AutoModelForCausalLM 
)
from datasets import load_dataset
from accelerate import Accelerator, dispatch_model, infer_auto_device_map
from accelerate import notebook_launcher # <-- CRUCIAL FOR KAGGLE
from accelerate.state import AcceleratorState # <-- For sharding data
from PIL import Image
import requests

# Set the global LLM choice
LLM_BACKBONE = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" 

# ==========================================
# 1. MULTI-STREAM INGESTOR (WITH MULTI-GPU SHARDING)
# ==========================================
class GeminiDataStreamer(IterableDataset):
    def __init__(self, mode="sft", max_length=128, max_audio_sec=5):
        self.mode = mode
        self.max_length = max_length
        self.max_audio_length = 16000 * max_audio_sec 
        
        self.tokenizer = AutoTokenizer.from_pretrained(LLM_BACKBONE)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        self.v_proc = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224", use_fast=True)
        self.a_proc = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")
        
        self.v_stream = load_dataset("HuggingFaceH4/llava-instruct-mix-vsft", split="train", streaming=True)
        self.a_stream = load_dataset("PolyAI/minds14", "en-US", split="train", streaming=True)
        
        if mode == "dpo":
            self.dpo_stream = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train", streaming=True)

        # MULTI-GPU FIX: Get current GPU index so we can shard the dataset
        state = AcceleratorState()
        self.process_index = state.process_index
        self.num_processes = state.num_processes

    def _extract_text(self, item):
        if isinstance(item, str): return item
        if isinstance(item, list) and len(item) > 0:
            if isinstance(item[0], dict) and 'text' in item[0]:
                return " ".join([i['text'] for i in item if 'text' in i])
        return str(item)

    def __iter__(self):
        v_iter = iter(self.v_stream)
        a_iter = iter(self.a_stream)
        d_iter = iter(self.dpo_stream) if self.mode == "dpo" else None
        
        step = 0
        while True:
            try:
                v_item = next(v_iter)
                a_item = next(a_iter)
                
                # MULTI-GPU SHARDING: GPU 0 gets even items, GPU 1 gets odd items!
                if step % self.num_processes != self.process_index:
                    step += 1
                    if self.mode == "dpo": next(d_iter) # Advance DPO iterator too
                    continue
                step += 1
                
                # Image
                img = v_item.get('image') or (v_item.get('images')[0] if v_item.get('images') else None)
                if img is None: continue
                pixels = self.v_proc(images=img.convert("RGB"), return_tensors="pt")['pixel_values'].squeeze(0)
                
                # Audio (Padded)
                audio_features = self.a_proc(
                    a_item['audio']['array'], 
                    sampling_rate=16000, 
                    padding="max_length", 
                    max_length=self.max_audio_length, 
                    truncation=True,
                    return_tensors="pt"
                )
                audio = audio_features['input_values'].squeeze(0)

                if self.mode == "sft":
                    raw_text = ""
                    if 'messages' in v_item:
                        raw_text = next((m['content'] for m in reversed(v_item['messages']) if m['role'] == 'assistant'), "")
                    elif 'conversations' in v_item:
                        raw_text = v_item['conversations'][-1]['value']
                    
                    text_str = self._extract_text(raw_text)
                    if not text_str: continue
                    
                    tokens = self.tokenizer(text_str, max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    yield pixels, tokens, audio
                
                else: # DPO
                    d_item = next(d_iter)
                    c_txt = d_item['chosen']
                    r_txt = d_item['rejected']
                    
                    c_tokens = self.tokenizer(c_txt, max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    r_tokens = self.tokenizer(r_txt, max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    
                    yield pixels, c_tokens, r_tokens, audio

            except StopIteration: break
            except Exception: continue


# ==========================================
# 2. SOTA ARCHITECTURE
# ==========================================
class SparseMoE(nn.Module):
    def __init__(self, d_model, num_experts=4):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model*2), nn.SiLU(), nn.Linear(d_model*2, d_model)) 
            for _ in range(num_experts)
        ])

    def forward(self, x):
        b, s, d = x.shape
        flat_x = x.view(-1, d)
        probs = F.softmax(self.gate(flat_x), dim=-1)
        weights, idx = torch.topk(probs, 1, dim=-1)
        
        out = torch.zeros_like(flat_x)
        for i, expert in enumerate(self.experts):
            mask = (idx == i).squeeze()
            if mask.any():
                out[mask] = expert(flat_x[mask]) * weights[mask]
        return out.view(b, s, d), probs.mean(0).std()


class GeminiLiteSOTA(nn.Module):
    def __init__(self, llm_name=LLM_BACKBONE, compute_dtype=torch.bfloat16):
        super().__init__()
        self.compute_dtype = compute_dtype
        
        # Load in BFloat16 natively
        self.llm = AutoModelForCausalLM.from_pretrained(llm_name, torch_dtype=compute_dtype)
        d_model = self.llm.config.hidden_size
        
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224", torch_dtype=compute_dtype)
        self.audio_enc = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base", torch_dtype=compute_dtype)
        
        self.llm.requires_grad_(False)
        self.vit.requires_grad_(False)
        self.audio_enc.requires_grad_(False)
        
        self.v_proj = nn.Linear(self.vit.config.hidden_size, d_model, dtype=compute_dtype)
        self.a_proj = nn.Linear(self.audio_enc.config.hidden_size, d_model, dtype=compute_dtype)
        self.moe = SparseMoE(d_model).to(compute_dtype)
        self.norm = nn.LayerNorm(d_model, dtype=compute_dtype)

    def forward(self, tokens, pixels, audio, pad_token_id):
        pixels = pixels.to(self.compute_dtype)
        audio = audio.to(self.compute_dtype)
        
        with torch.no_grad():
            v_feat = self.vit(pixels).last_hidden_state
            a_feat = self.audio_enc(audio).last_hidden_state
        
        t_embeds = self.llm.get_input_embeddings()(tokens).to(self.compute_dtype)
        
        v_embeds = self.v_proj(v_feat)
        a_embeds = self.a_proj(a_feat)
        
        mm_embeds = torch.cat([a_embeds, v_embeds], dim=1)
        mm_moe_out, aux_loss = self.moe(self.norm(mm_embeds))
        mm_embeds = mm_embeds + mm_moe_out 
        
        inputs_embeds = torch.cat([mm_embeds, t_embeds], dim=1)
        
        batch_size = tokens.size(0)
        mm_len = mm_embeds.size(1)
        
        mm_mask = torch.ones((batch_size, mm_len), dtype=torch.long, device=tokens.device)
        text_mask = (tokens != pad_token_id).long()
        attention_mask = torch.cat([mm_mask, text_mask], dim=1)
        
        outputs = self.llm(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
        
        return outputs.logits, aux_loss, mm_len


# ==========================================
# 3. LOSS FUNCTIONS
# ==========================================
def compute_causal_loss(logits, tokens, mm_len, pad_token_id):
    labels = tokens.clone()
    labels[labels == pad_token_id] = -100
    batch_size = tokens.size(0)
    mm_labels = torch.full((batch_size, mm_len), -100, dtype=torch.long, device=tokens.device)
    full_labels = torch.cat([mm_labels, labels], dim=1)
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = full_labels[..., 1:].contiguous()
    return F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

def get_batch_logps(logits, tokens, mm_len, pad_token_id):
    labels = tokens.clone()
    labels[labels == pad_token_id] = -100
    mm_labels = torch.full((tokens.size(0), mm_len), -100, dtype=torch.long, device=tokens.device)
    full_labels = torch.cat([mm_labels, labels], dim=1)
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = full_labels[..., 1:].contiguous()
    log_probs = F.log_softmax(shift_logits, dim=-1)
    loss_mask = shift_labels != -100
    shift_labels[shift_labels == -100] = 0 
    token_log_probs = torch.gather(log_probs, dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)
    return (token_log_probs * loss_mask).sum(dim=-1)


# ==========================================
# 4. DISTRIBUTED TRAINING EXECUTION
# ==========================================
def train():
    # Accelerate wraps everything for DDP natively
    acc = Accelerator(mixed_precision="bf16")
    
    # We use acc.print instead of print so we don't get double prints from 2 GPUs
    acc.print(">>> Initializing Models and Tokenizers...")
    
    tokenizer = AutoTokenizer.from_pretrained(LLM_BACKBONE)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = GeminiLiteSOTA(compute_dtype=torch.bfloat16)
    
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable_params, lr=1e-4)
    
    acc.print(">>> Phase 1: SFT (Multimodal Alignment)...")
    # Batch size of 1 per GPU = Effective batch size of 2 across your T4 x2
    sft_loader = DataLoader(GeminiDataStreamer(mode="sft"), batch_size=1)
    
    model, opt, sft_loader = acc.prepare(model, opt, sft_loader)
    
    model.train()
    for i, (p, t, a) in enumerate(sft_loader):
        opt.zero_grad()
        logits, aux_loss, mm_len = model(t, p, a, tokenizer.pad_token_id)
        loss = compute_causal_loss(logits, t, mm_len, tokenizer.pad_token_id)
        
        total_loss = loss + 0.01 * aux_loss
        acc.backward(total_loss)
        opt.step()
        
        if i % 10 == 0: 
            acc.print(f"SFT Step {i} | CrossEntropy Loss: {loss.item():.4f}")
        if i >= 100: break # Halved steps because effective batch size is doubled

    acc.print("\n>>> Phase 2: DPO (Preference Alignment)...")
    dpo_loader = DataLoader(GeminiDataStreamer(mode="dpo"), batch_size=1)
    dpo_loader = acc.prepare(dpo_loader)
    
    for i, (p, c_t, r_t, a) in enumerate(dpo_loader):
        opt.zero_grad()
        
        logits_c, _, mm_len = model(c_t, p, a, tokenizer.pad_token_id)
        logits_r, _, _ = model(r_t, p, a, tokenizer.pad_token_id)
        
        l_c = get_batch_logps(logits_c, c_t, mm_len, tokenizer.pad_token_id)
        l_r = get_batch_logps(logits_r, r_t, mm_len, tokenizer.pad_token_id)
        
        dpo_loss = -F.logsigmoid(0.1 * (l_c - l_r)).mean()
        
        acc.backward(dpo_loss)
        opt.step()
        
        if i % 10 == 0: 
            acc.print(f"DPO Step {i} | Loss: {dpo_loss.item():.4f}")
        if i >= 50: break

    acc.print("\n>>> Training Finished Successfully!")
    
    # Sync both GPUs before saving
    acc.wait_for_everyone()
    if acc.is_main_process:
        unwrap_model = acc.unwrap_model(model)
        trainable_weights = {
            "v_proj": unwrap_model.v_proj.state_dict(),
            "a_proj": unwrap_model.a_proj.state_dict(),
            "moe": unwrap_model.moe.state_dict(),
        }
        torch.save(trainable_weights, "gemini_adapters.pth")
        acc.print("Multimodal adapters saved to gemini_adapters.pth")


# ==========================================
# 5. MULTI-GPU INFERENCE FUNCTION
# ==========================================
@torch.no_grad()
def generate_response(model, tokenizer, question, image_path_or_url, max_new_tokens=30):
    model.eval()
    
    device = next(model.parameters()).device
    compute_dtype = model.compute_dtype
    
    v_proc = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
    a_proc = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")

    if image_path_or_url.startswith("http"):
        img = Image.open(requests.get(image_path_or_url, stream=True).raw).convert("RGB")
    else:
        img = Image.open(image_path_or_url).convert("RGB")
    
    pixels = v_proc(images=img, return_tensors="pt")['pixel_values'].to(device, dtype=compute_dtype)
    audio = a_proc(torch.zeros(16000 * 5), sampling_rate=16000, return_tensors="pt")['input_values'].to(device, dtype=compute_dtype)
    
    prompt = f"<|user|>\n<image>\n{question}</s>\n<|assistant|>\n"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    v_feat = model.vit(pixels).last_hidden_state
    a_feat = model.audio_enc(audio).last_hidden_state
    v_embeds = model.v_proj(v_feat)
    a_embeds = model.a_proj(a_feat)
    
    mm_embeds = torch.cat([a_embeds, v_embeds], dim=1)
    mm_moe_out, _ = model.moe(model.norm(mm_embeds))
    mm_embeds = mm_embeds + mm_moe_out
    
    t_embeds = model.llm.get_input_embeddings()(input_ids).to(compute_dtype)
    inputs_embeds = torch.cat([mm_embeds, t_embeds], dim=1)
    
    mm_mask = torch.ones((1, mm_embeds.size(1)), dtype=torch.long, device=device)
    t_mask = torch.ones((1, input_ids.size(1)), dtype=torch.long, device=device)
    attention_mask = torch.cat([mm_mask, t_mask], dim=1)
    
    generated_ids = model.llm.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.7,
        do_sample=True
    )
    
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)


if __name__ == "__main__":
    # =========================================================
    # KAGGLE MAGIC: This spawns 2 processes natively!
    # =========================================================
    print("Launching Distributed Training on 2 GPUs...")
    notebook_launcher(train, num_processes=2)
    
    print("\n--- Running Multi-GPU Inference Test ---")
    tok = AutoTokenizer.from_pretrained(LLM_BACKBONE)
    
    # 1. Load the model framework in bfloat16 to CPU first
    test_model = GeminiLiteSOTA(compute_dtype=torch.bfloat16)
    
    # 2. Load the trained adapters
    checkpoint = torch.load("gemini_adapters.pth", map_location="cpu")
    test_model.v_proj.load_state_dict(checkpoint['v_proj'])
    test_model.a_proj.load_state_dict(checkpoint['a_proj'])
    test_model.moe.load_state_dict(checkpoint['moe'])
    
    # 3. SOTA Multi-GPU Dispatch for Inference (Pipeline Parallelism)
    # This automatically splits the model's memory evenly across GPU 0 and GPU 1
    device_map = infer_auto_device_map(test_model, max_memory={0: "14GiB", 1: "14GiB"})
    test_model = dispatch_model(test_model, device_map=device_map)
    
    test_image = "/kaggle/input/datasets/anwarshamim/images-for-test/images.jpg"
    
    answer = generate_response(test_model, tok, "Describe this image:", test_image)
    print(f"\nMODEL RESPONSE:\n{answer}")

Launching Distributed Training on 2 GPUs...


ValueError: To launch a multi-device training from your notebook, the `Accelerator` should only be initialized inside your training function. Restart your notebook and make sure no cells initializes an `Accelerator`.